# Lean-41 : Capstone — escalier vers la sous-série MIMO (détection par flips, budget de descente et digestion)

**Série** : [SymbolicAI/Lean](README.md) — entrée/sortie de la sous-série [MIMO](MIMO/README.md).

**Navigation** : [<< Lean-40 Capstone Langlands](Lean-40-Capstone-Langlands-Python.ipynb) | [Index](README.md)

**Carnets visés** : `MIMO-01-Descente-Budget-Lean-Python` → `MIMO-04-Capstone-Digestion-Python` (quatre carnets), avec leur lac compagnon [`mimo_lean/`](mimo_lean/README.md) (0 sorry, kernel `lean4-wsl`).

**Pourquoi un escalier** : la sous-série MIMO déplace 21c/21/21b dans un dossier `MIMO/` ([PR #20371](https://github.com/jsboige/CoursIA/pull/20371), EPIC #17545). Sans carnet d'entrée, la descente est inatteignable depuis la série principale. Ce capstone suit la convention tenue par Lean-20 (vers `ANALYSE/`), Lean-37 (vers `Serre100/`), Lean-38 (vers `Geometry/`), Lean-39 (vers `GOL/`) et Lean-40 (vers `Langlands/`) : table de routage, prérequis, surface mesurée, et exercices qui confrontent la promesse à l'exécution.


## 1. Le geste : la détection MIMO par flips de coordonnées

Papailiopoulos (2026) pose la détection MIMO comme une **descente sur un paysage de coût** : à chaque pas, on flippe la coordonnée la plus prometteuse, on mesure la décroissance du coût, et on s'arrête quand la décroissance n'est plus garantie par le budget restant. Le seuil `2·log N` (en bits) provient de l'union bound sur la queue gaussienne de l'estimateur LMMSE.

Trois usages de Lean/IA cohabitent ici, comme dans Lean-20 :

1. **Vérifier** : `#check` des énoncés du lac — sont-ils prouvés, ou empruntés ?
2. **Raconter** : la sous-série est une narration incarnée du seuil `2·log N`, de l'instanciation à la digestion.
3. **Extraire** : MIMO-04 exécute les énoncés à la main sur une instance réelle — c'est la sortie, pas l'entrée.

Ce capstone pose la question d'entrée : *par où commencer, et que faut-il savoir pour ne pas se perdre ?*


## 2. Table de routage : quatre carnets, un lac

| Carnet | Noyau | Thème | Prérequis | Sortie |
|---|---|---|---|---|
| [MIMO-01-Descente-Budget-Lean-Python](MIMO/MIMO-01-Descente-Budget-Lean-Python.ipynb) | `python3` + Lean citation | Le budget : la décroissance du coût, le seuil `2·log N`, le théorème `descent_target_before_ceiling`, les trois cas de violation de `hstrict` | Aucun formel | Diagnostic de la descente, mesure des sources du lac |
| [MIMO-02-Detection-Flips-Lean-Python](MIMO/MIMO-02-Detection-Flips-Lean-Python.ipynb) | `python3` + Lean citation | Le problème réel : détection MIMO, Monte-Carlo vs `e^{−np}`, `#check` des 4 phases + `ml_error_prob_ge_threshold` | Lean 1-6 (tutoriel) | Confrontation empirique/asymptotique |
| [MIMO-03-Converse-Native-Lean](MIMO/MIMO-03-Converse-Native-Lean.ipynb) | `lean4-wsl` | Le lac **exécuté en kernel Lean réel** : frontière SLT exhibée, modules visités énoncé par énoncé, `#print axioms` | Lean 1-6 + dispositions à lire des `.lean` | Confirmation que les énoncés sont prouvés (pas empruntés) |
| [MIMO-04-Capstone-Digestion-Python](MIMO/MIMO-04-Capstone-Digestion-Python.ipynb) | `python3` | Les énoncés exécutés à la main : cinq faits de `Descent.lean`, témoin `DescentFailure` sur identité sans bruit, queue `NormTails` Monte-Carlo, surface du lac recomptée | NumPy seulement | Sortie de la sous-série, mesure vs promesse |

**Itinéraire recommandé** : `MIMO-01` (lire l'algorithme nu) → `MIMO-02` (le seuil et la simulation) → `MIMO-03` (le lac formel) → `MIMO-04` (la digestion). Les flèches de retour depuis `MIMO-04` : ce que Lean prouve, ce que Python mesure, et où le pont s'établit (ou pas).


## 3. Le versant formel : `mimo_lean`, un lac sorry-free

Le lac [`mimo_lean/`](mimo_lean/README.md) (baseline 0 sorry en code-only) est entièrement **own** sauf la concentration de Lipschitz et Hanson–Wright, empruntées au fork [`jsboige/lean-stat-learning-theory`](https://github.com/jsboige/lean-stat-learning-theory) (lui-même un fork de `YuanheZ/lean-stat-learning-theory`). Les déclarations empruntées sont marquées par leur chemin d'import ; tout le reste — l'instanciation MIMO, le converse chi-carré, le pont vers l'erreur ML — est prouvé ici.

Cibles du lac par carnet :

- `MIMO-01` expose `Descent` seul (l'algorithme, sans dépendance Mathlib).
- `MIMO-02` expose `Objective`, `Lmmse`, la glue `Converse`, `Bridge` (partiel) et `NormTails`.
- `MIMO-03` montre le lac **en entier** : `#check` puis `#print axioms` sur chaque module.
- `MIMO-04` **recompte** la surface du lac depuis les sources — c'est le test de cohérence (l'organe canonique rend 56, là où les sources en portent 57 — voir §5).


In [1]:
# Prérequis minimaux : aucun pour ce capstone d'entrée.
import sys, platform
import numpy as np

print(f"Python: {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"NumPy:  {np.__version__}")

# Vérification que la sous-série est bien atteignable depuis ce carnet.
from pathlib import Path
serie_root = Path('MIMO')
assert serie_root.exists(), "Le dossier MIMO/ doit être sibling de ce carnet"

carnets = sorted(serie_root.glob('MIMO-*.ipynb'))
assert len(carnets) >= 4, f"Quatre carnets attendus, {len(carnets)} trouves"
for c in carnets:
    print(f"  {c.name}: {c.stat().st_size:>6} octets")

# Vérification que le lac compagnon existe.
lac = Path('mimo_lean')
assert lac.exists(), f"Le lac {lac} doit etre sibling du dossier Lean/"
lean_files = sorted(lac.rglob('*.lean'))
print(f"\nLac mimo_lean: {len(lean_files)} fichiers .lean")


Python: 3.13.15 (CPython)
NumPy:  2.5.2
  MIMO-01-Descente-Budget-Lean-Python.ipynb: 114817 octets
  MIMO-02-Detection-Flips-Lean-Python.ipynb: 164359 octets
  MIMO-03-Converse-Native-Lean.ipynb: 351827 octets
  MIMO-04-Capstone-Digestion-Python.ipynb:  48942 octets

Lac mimo_lean: 13 fichiers .lean


### Lecture du résultat

Les quatre carnets de la sous-série `MIMO/` sont présents, ainsi que leur lac `mimo_lean/`. Aucune autre dépendance n'est requise pour le présent carnet d'entrée — un kernel `python3` standard suffit.


## 4. Surface de la sous-série : on mesure, on ne suppose rien

Un capstone sert à **décider si l'on entre**, et cette décision se prend sur des mesures, pas sur des promesses. La surface de la sous-série MIMO est ici recomptée depuis les sources du dépôt :

- nombre de carnets, lignes de code, exercices ;
- taille du lac, modules, énoncés `theorem` / `lemma` ;
- écart entre promesse (énoncés dans le README du lac) et un comptage direct (un `grep` sur `mimo_lean/` rend 57 là où l'**organe** `scripts/lean/scan_lake_notebook_visibility.py` en rend 56 — voir §5).


In [2]:
# Surface de la sous-serie : on lit les sources, on ne declare rien.
from pathlib import Path
import re

# --- 4 carnets de la sous-serie MIMO
carnets = sorted(Path('MIMO').glob('MIMO-*.ipynb'))
import json
total_cells = total_code = total_md = total_exos = 0
for p in carnets:
    nb = json.loads(p.read_text(encoding='utf-8'))
    cells = nb.get('cells', [])
    n_code = sum(1 for c in cells if c['cell_type'] == 'code')
    n_md = sum(1 for c in cells if c['cell_type'] == 'markdown')
    # Heuristique exercice : en-tete "Exercice" dans une cellule markdown
    n_exos = sum(
        1 for c in cells
        if c['cell_type'] == 'markdown'
        and any('Exercice' in line for line in c.get('source', []))
    )
    total_cells += len(cells)
    total_code += n_code
    total_md += n_md
    total_exos += n_exos
    print(f"  {p.name:54s} cells={len(cells):3d}  code={n_code:2d}  md={n_md:2d}  exos={n_exos}")

print(f"\nSous-serie MIMO : {len(carnets)} carnets, {total_cells} cellules "
      f"({total_code} code, {total_md} markdown), {total_exos} exercices")


  MIMO-01-Descente-Budget-Lean-Python.ipynb              cells= 24  code= 9  md=15  exos=6
  MIMO-02-Detection-Flips-Lean-Python.ipynb              cells= 41  code=13  md=28  exos=5
  MIMO-03-Converse-Native-Lean.ipynb                     cells= 34  code=16  md=18  exos=1
  MIMO-04-Capstone-Digestion-Python.ipynb                cells= 26  code= 9  md=17  exos=3

Sous-serie MIMO : 4 carnets, 125 cellules (47 code, 78 markdown), 15 exercices


In [3]:
# Surface du lac mimo_lean : on invoque l'organe canonique
# `scripts/lean/count_code_sorry.py` (le seul validant le compte global
# du lake). On n'en reimplemente pas la regex ici : l'instrument tient
# deja compte du dedoublonnage i18n FR/EN (siblings #4980) et de la
# distinction code-only / prose. Cf. [anti-regression.md §instruments]
# pour la discussion des faux negatifs/symetriques d'un motif artisanal.
import json as _json
import subprocess
from pathlib import Path

# Le cwd du kernel est le dossier du carnet (MyIA.AI.Notebooks/SymbolicAI/Lean/).
# On retrouve la racine du depot par ancrage sur ce carnet, puis on passe
# au script un chemin ABSOLU vers le lac (l'organe resout les chemins
# relativement a son cwd, pas a celui de l'appelant).
_nb_dir = Path('.').resolve()
_repo_root = _nb_dir
for _ in range(6):
    if (_repo_root / 'scripts' / 'lean' / 'count_code_sorry.py').exists():
        break
    _repo_root = _repo_root.parent
script = _repo_root / 'scripts' / 'lean' / 'count_code_sorry.py'
assert script.exists(), f"count_code_sorry.py introuvable : recherche depuis {_nb_dir}"

# Lac mimo_lean (chemin absolu)
lac_abs = (_repo_root / 'MyIA.AI.Notebooks' / 'SymbolicAI' / 'Lean' / 'mimo_lean').resolve()
assert lac_abs.exists(), f"Lac mimo_lean introuvable : {lac_abs}"
lac_rel = Path('mimo_lean')  # pour le listing
lean_files = sorted(p for p in lac_rel.rglob('*.lean'))
print(f"Fichiers .lean trouves dans le lac : {len(lean_files)}")

# Invocation unique de l'organe canonique sur le lac entier
proc = subprocess.run(
    ['python', str(script), '--lake', str(lac_abs), '--json', '--no-vacuous'],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
data = _json.loads(proc.stdout)
lake_summary = data['lakes'][0]

n_files = lake_summary['files']
n_naive = lake_summary['naive_sorry']
n_code = lake_summary['code_sorry']
n_distinct = lake_summary['distinct_code_sorry']

print()
print(f"Lac mimo_lean : {n_files} fichiers .lean, naive_sorry={n_naive}, code_sorry={n_code}, distinct_code_sorry={n_distinct} (i18n dedoublonne)")
print("Le champ `distinct_code_sorry` est l'unique mesure de reference pour la dette formelle.")
print("`naive_sorry` sur-compte la prose ; `code_sorry` filtre les commentaires.")
print("Le dedoublonnage FR/EN (siblings #4980) est applique automatiquement.")

# Verification baseline
assert n_distinct == 0, (
    f"mimo_lean devrait etre sorry-free (organe canonique distinct_code_sorry=0), "
    f"{n_distinct} trouves"
)
print()
print("Baseline mimo_lean OK : 0 sorry code-only, 0 vacuous -- le lac est la base formelle de la sous-serie.")


Fichiers .lean trouves dans le lac : 13



Lac mimo_lean : 13 fichiers .lean, naive_sorry=4, code_sorry=0, distinct_code_sorry=0 (i18n dedoublonne)
Le champ `distinct_code_sorry` est l'unique mesure de reference pour la dette formelle.
`naive_sorry` sur-compte la prose ; `code_sorry` filtre les commentaires.
Le dedoublonnage FR/EN (siblings #4980) est applique automatiquement.

Baseline mimo_lean OK : 0 sorry code-only, 0 vacuous -- le lac est la base formelle de la sous-serie.


### Lecture du résultat

La sous-série `MIMO/` porte un carnet par marche de l'escalier (le minimum conventionnel de la série) ; le lac `mimo_lean/` est **sorry-free** (baseline 0, vérifiée par `lean-mimo.yml`). Les chiffres exacts varieront au fil des PRs — c'est l'**écart** entre ce que cet organe compte et ce que les autres (README du lac, scripts/lean) comptent qui fait foi (cf. §5).


## 5. L'écart promesse / organe : `56` vs `57`

L'écart 56 / 57 est **réel**, et il est instrumenté : l'organe canonique `scripts/lean/scan_lake_notebook_visibility.py --lake mimo_lean` rend `56` là où un comptage direct des sources en rend `57`. La cause est dans son motif de nom : `DECL_RE` capture `([A-Za-z_][A-Za-z0-9_']*)`, donc la capture s'arrête au **premier caractère non ASCII**. (Le script `scripts/lean/extract_decls.py` que citait un premier récit n'existe pas dans l'arbre : `git ls-tree -r origin/main --name-only | grep extract_decls` rend 0 hit.)

**Le 57ᵉ n'est pas un nom commençant par un caractère grec** (Π ou Θ), comme on a pu le croire. C'est `B_ρ` : un `noncomputable def` (donc pas un `theorem` / `lemma`), dans `MyIA.AI.Notebooks/SymbolicAI/Lean/mimo_lean/Lmmse.lean:84`. Le motif s'arrêtant au premier caractère non ASCII, `B_ρ` est capturé comme `B_` — exactement comme `B_ρ_posSemidef` (`Lmmse.lean:90`) : les deux déclarations se replient sur le même nom, et le compte en perd une. Recensement exhaustif des fichiers du lac : aucun nom de déclaration ne commence par un caractère non-ASCII.

**Le 56 / 57 n'est pas mesuré dans ce carnet.** La cellule §4 ne compte que les carnets, lignes de code, exercices et la baseline sorry du lac (`mimo_lean` : 0 `distinct_code_sorry`, vérifié par `lean-mimo.yml`). Le compte de déclarations theorem / lemma / def du lac, lui, vit dans **MIMO-04** (la mesure sœur), qui porte en cellule c17 le tableau détaillé par module (`Descent 7+3`, `Objective 4+2`, `Bridge 13+2`, `Converse 16`, `NormTails 6`, `Lmmse 3+1` — où le `+1` final de Lmmse est précisément `B_ρ`). Pour qui entre dans la sous-série : **lire MIMO-04 c17 pour la mesure brute**, et ne pas chercher 56 ou 57 dans les sorties committées du présent carnet — elles n'y sont pas.

La leçon tient : **ne pas se fier au seul chiffre d'un organe**. Croiser avec un `grep` direct sur `mimo_lean/` pour la mesure brute, citer l'instrument **réellement présent dans l'arbre** pour la mesure reproductible. L'écart 56 / 57 est documenté dans la PR de digestion de MIMO-04.


## 6. Itinéraire de lecture : trois parcours au choix

Trois parcours sont possibles, selon le temps et la curiosité :

1. **Pressé (1h)** : `MIMO-01` (l'algorithme et le budget) puis `MIMO-04` (la sortie). On quitte avec une vue de la détection par flips et de sa digestion.
2. **Curieux (2h30)** : `MIMO-01` → `MIMO-02` (le seuil et la simulation). On quitte en ayant vu le seuil `2·log N` confrontée au Monte-Carlo.
3. **Formel (3h+)** : `MIMO-01` → `MIMO-02` → `MIMO-03` (le lac en kernel Lean) → `MIMO-04` (la digestion). On quitte en ayant vu la boucle complète : algorithme, simulation, preuve, et re-mesure.

Le parcours principal reprend ensuite à [Lean-22 — Galois](GALOIS/GALOIS-01-Probleme-Inverse-M23-Python.ipynb) (sous-série Galois) ou à [Lean-29 — Hecke Operators](Lean-29-Hecke-Operators-Native.ipynb) (sous-série Hecke).


## 7. Trois exercices

**Exercice 1 — reproduire le seuil `2·log N`** : reprendre la cellule Monte-Carlo de MIMO-02 et vérifier que la probabilité d'échappement du bruit suit `e^{−n·p}` à `n` croissant pour `p = 0.05`, `0.1`, `0.2`. Tracer la courbe et citer le seuil en bits.

**Exercice 2 — compter les déclarations par module** : à partir de la cellule §4, compter les `theorem` et `lemma` par module de `mimo_lean/` (`Descent`, `Objective`, `Lmmse`, `Converse`, `Bridge`, `NormTails`). Lequel a le plus d'énoncés ? Le moins ?

**Exercice 3 — l'écart 56 / 57 par caractère non-ASCII en position quelconque** : combien de déclarations du lac ont un nom **contenant** un caractère non-ASCII (pas seulement en début) ? Écrire un script qui les liste, et vérifier que la liste contient bien `B_ρ` (ρ en 3ᵉ position, `Lmmse.lean:84`, déclaré `noncomputable def`). *Indice* : un simple `not c.isascii()` sur les caractères du nom suffit — `unicodedata.category(c)[0] != 'L'` ne marche pas ici, parce que `category('ρ') = 'Ll'` (lettre minuscule) passerait le test comme une lettre ASCII.
